In [2]:
import torch
import librosa
import soundfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn

print(torch.__version__)
print(torch.cuda.is_available())

from google.colab import files

uploaded = files.upload()
uploaded = files.upload()
uploaded = files.upload()
uploaded = files.upload()
uploaded = files.upload()
uploaded = files.upload()
uploaded = files.upload()
uploaded = files.upload()
uploaded = files.upload()
uploaded = files.upload()

audio_files = ["AuSep_1_fl_14_Waltz.wav", "AuSep_1_vn_02_Sonata.wav", "AuSep_2_vn_19_Pavane.wav", "AuSep_1_vn_17_Nocturne.wav", "AuSep_1_vn_12_Spring.wav"]

notation_files = ["Waltz.csv", "Sonata.csv", "Pavane.csv", "Nocturne.csv", "Spring.csv"]

print("Detected audio files:", audio_files)
print("Detected notation files:", notation_files)


class MusicDataset(Dataset):
    def __init__(self, audio_files, label_files):
        self.audio_files = audio_files
        self.label_files = label_files

    def __len__(self):
        return len(self.audio_files)

    def __getitem__(self, index):
        audio_path = self.audio_files[index]
        label_path = self.label_files[index]
        y, sr = librosa.load(audio_path, sr=None)
        D = librosa.stft(y)
        S_db = librosa.amplitude_to_db(np.abs(D), ref=np.max)
        spectrogram_tensor = torch.tensor(S_db, dtype=torch.float32)
        labels = pd.read_csv(label_path,  header=None)

        label_tensor = torch.tensor(labels.values, dtype=torch.float32)
        return spectrogram_tensor, label_tensor

dataset=MusicDataset(audio_files, notation_files)
spectrogram, labels=dataset[0]
print(spectrogram.shape)
print(labels.shape)
print(labels[:5])

loader = DataLoader(dataset, batch_size=1, shuffle=True)

for spectrogram, labels in loader:
    print("Batch spectrogram shape:", spectrogram.shape)
    print("Batch labels shape:", labels.shape)
    break

class CRNN(nn.Module):
  def __init__(self, num_freq_bins=1025, cnn_channels=16, hidden_size=64, num_classes=10):
    super(CRNN, self).__init__()

    self.conv1=nn.Conv2d(in_channels=1, out_channels=cnn_channels, kernel_size=3, padding=1)
    self.relu1=nn.ReLU()
    self.pool1=nn.MaxPool2d(kernel_size=2)

    self.lstm=nn.LSTM(input_size=cnn_channels*(num_freq_bins//2), hidden_size=hidden_size, num_layers=2, batch_first=True)
    self.fc=nn.Linear(hidden_size, num_classes)

  def forward(self, x):
    x=self.conv1(x)
    x=self.relu1(x)
    x=self.pool1(x)

    batch_size, channels, freq, time=x.shape
    x=x.permute(0,3,1,2)
    x=x.reshape(batch_size, time, channels*freq)

    x, _=self.lstm(x)
    x=self.fc(x)

    return x

model=CRNN()

2.11.0+cpu
False


Saving AuSep_1_fl_14_Waltz.wav to AuSep_1_fl_14_Waltz (1).wav


Saving AuSep_1_vn_02_Sonata.wav to AuSep_1_vn_02_Sonata (1).wav


Saving AuSep_2_vn_19_Pavane.wav to AuSep_2_vn_19_Pavane (1).wav


Saving AuSep_1_vn_17_Nocturne.wav to AuSep_1_vn_17_Nocturne (1).wav


...

Saving Waltz.csv to Waltz (1).csv


Saving Sonata.csv to Sonata (1).csv


Saving Pavane.csv to Pavane (1).csv


Saving Nocturne.csv to Nocturne (1).csv


Saving Spring.csv to Spring (1).csv
Detected audio files: ['AuSep_1_fl_14_Waltz.wav', 'AuSep_1_vn_02_Sonata.wav', 'AuSep_2_vn_19_Pavane.wav', 'AuSep_1_vn_17_Nocturne.wav', 'AuSep_1_vn_12_Spring.wav']
Detected notation files: ['Waltz.csv', 'Sonata.csv', 'Pavane.csv', 'Nocturne.csv', 'Spring.csv']
torch.Size([1025, 8700])
torch.Size([294, 5])
tensor([[1.4260e+00,        nan, 6.3293e+02,        nan, 8.4200e-01],
        [2.2740e+00,        nan, 6.0017e+02,        nan, 8.1900e-01],
        [3.0980e+00,        nan, 6.2928e+02,        nan, 6.1500e-01],
        [3.7250e+00,        nan, 5.2902e+02,        nan, 2.6700e-01],
        [3.9980e+00,        nan, 5.9692e+02,        nan, 2.5500e-01]])
Batch spectrogram shape: torch.Size([1, 1025, 12504])
Batch labels shape: torch.Size([1, 211, 5])
